# Vanguard A/B Testing Project
## 01 — Data Discovery, Cleaning & Experiment Analysis

This notebook contains the cleaned analytical workflow for the Vanguard A/B testing project.

### Main questions
1. Did the Test version improve the completion rate?
2. Did the Test version meet the required 5% improvement threshold?
3. How much time did users spend at each process step?
4. Did the Test version create more navigation errors?
5. Were the Test and Control groups sufficiently comparable?
6. Who are the clients using the digital process?

The final tables required for Tableau are exported to `../data/processed/`.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import ttest_ind, chi2_contingency, norm
from statsmodels.stats.proportion import proportions_ztest

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")


## 1. Load the raw datasets

In [ ]:
demo = pd.read_csv("../data/raw/df_final_demo.txt", sep=",")
experiment_clients = pd.read_csv(
    "../data/raw/df_final_experiment_clients.txt",
    sep=","
)
web_data_pt1 = pd.read_csv(
    "../data/raw/df_final_web_data_pt_1.txt",
    sep=","
)
web_data_pt2 = pd.read_csv(
    "../data/raw/df_final_web_data_pt_2.txt",
    sep=","
)

print("Demo:", demo.shape)
print("Experiment clients:", experiment_clients.shape)
print("Web part 1:", web_data_pt1.shape)
print("Web part 2:", web_data_pt2.shape)


## 2. Data quality checks

In [ ]:
print("=== MISSING VALUES ===")
print("\nDemo:")
print(demo.isna().sum())

print("\nExperiment clients:")
print(experiment_clients.isna().sum())

print("\nWeb part 1:")
print(web_data_pt1.isna().sum())

print("\nWeb part 2:")
print(web_data_pt2.isna().sum())

print("\n=== DUPLICATES ===")
print("Demo:", demo.duplicated().sum())
print("Experiment clients:", experiment_clients.duplicated().sum())
print("Web part 1:", web_data_pt1.duplicated().sum())
print("Web part 2:", web_data_pt2.duplicated().sum())


In [ ]:
# Remove exact duplicate web events.
web_data_pt1_clean = web_data_pt1.drop_duplicates().copy()
web_data_pt2_clean = web_data_pt2.drop_duplicates().copy()

web_data = pd.concat(
    [web_data_pt1_clean, web_data_pt2_clean],
    ignore_index=True
)

print("Clean web data shape:", web_data.shape)
print("Unique clients:", web_data["client_id"].nunique())
print("Remaining duplicates:", web_data.duplicated().sum())


In [ ]:
# Convert timestamps to datetime.
web_data["date_time"] = pd.to_datetime(
    web_data["date_time"],
    errors="coerce"
)

print("Invalid dates:", web_data["date_time"].isna().sum())
print("Date range:", web_data["date_time"].min(), "to", web_data["date_time"].max())
print("Process steps:", sorted(web_data["process_step"].dropna().unique()))


## 3. Identify the A/B experiment population

In [ ]:
# Keep only clients with a known Test/Control assignment.
experiment_clients_ab = experiment_clients[
    experiment_clients["Variation"].isin(["Test", "Control"])
].copy()

print(experiment_clients_ab["Variation"].value_counts())
print("A/B clients:", experiment_clients_ab["client_id"].nunique())


In [ ]:
# Merge web interactions with experiment assignment.
web_with_variation = web_data.merge(
    experiment_clients_ab[["client_id", "Variation"]],
    on="client_id",
    how="inner"
)

print("Experiment web data:", web_with_variation.shape)
print(web_with_variation["Variation"].value_counts())


## 4. Funnel and completion rate

In [ ]:
# Define a client as converted when they reach the confirm step.
client_conversion = (
    web_with_variation
    .groupby(["client_id", "Variation"])["process_step"]
    .apply(lambda steps: "confirm" in steps.values)
    .reset_index(name="converted")
)

ab_conversion = (
    client_conversion
    .groupby("Variation")["converted"]
    .agg(
        clients="size",
        conversions="sum"
    )
)

ab_conversion["conversion_rate_%"] = (
    ab_conversion["conversions"]
    / ab_conversion["clients"]
    * 100
).round(2)

ab_conversion


In [ ]:
# Statistical comparison of Test vs Control conversion rates.
conversions = ab_conversion["conversions"].values
clients = ab_conversion["clients"].values

z_conversion, p_conversion = proportions_ztest(
    conversions,
    clients
)

print("Z-statistic:", round(z_conversion, 4))
print("P-value:", p_conversion)


In [ ]:
control_rate = (
    ab_conversion.loc["Control", "conversions"]
    / ab_conversion.loc["Control", "clients"]
)

test_rate = (
    ab_conversion.loc["Test", "conversions"]
    / ab_conversion.loc["Test", "clients"]
)

absolute_improvement_pp = (test_rate - control_rate) * 100
relative_improvement_pct = (test_rate / control_rate - 1) * 100

print("Control completion rate:", round(control_rate * 100, 2), "%")
print("Test completion rate:", round(test_rate * 100, 2), "%")
print("Absolute improvement:", round(absolute_improvement_pp, 2), "percentage points")
print("Relative improvement:", round(relative_improvement_pct, 2), "%")


## 5. Required 5% improvement threshold

In [ ]:
threshold_rate = control_rate * 1.05

# One-sided test: is the Test rate greater than the 5% threshold?
test_n = ab_conversion.loc["Test", "clients"]

z_threshold = (
    test_rate - threshold_rate
) / np.sqrt(
    test_rate * (1 - test_rate) / test_n
)

p_threshold = 1 - norm.cdf(z_threshold)

print("5% threshold:", round(threshold_rate * 100, 2), "%")
print("Test completion rate:", round(test_rate * 100, 2), "%")
print("Z-statistic:", round(z_threshold, 4))
print("One-sided p-value:", round(p_threshold, 6))


## 6. Time spent at each process step

In [ ]:
time_data = web_with_variation.copy()
time_data["date_time"] = pd.to_datetime(time_data["date_time"])

time_data = time_data.sort_values(
    ["visit_id", "date_time"]
)

time_data["next_step"] = (
    time_data.groupby("visit_id")["process_step"].shift(-1)
)

time_data["next_time"] = (
    time_data.groupby("visit_id")["date_time"].shift(-1)
)

time_data["time_spent_seconds"] = (
    time_data["next_time"] - time_data["date_time"]
).dt.total_seconds()

# The final event in a visit has no following event, so it is excluded.
time_data = time_data.dropna(subset=["time_spent_seconds"]).copy()

average_time_by_step = (
    time_data
    .groupby(["Variation", "process_step"])["time_spent_seconds"]
    .mean()
    .reset_index()
)

average_time_by_step["time_spent_minutes"] = (
    average_time_by_step["time_spent_seconds"] / 60
).round(2)

average_time_by_step


## 7. Navigation error rate

In [ ]:
error_data = web_with_variation.copy()
error_data["date_time"] = pd.to_datetime(error_data["date_time"])

error_data = error_data.sort_values(
    ["visit_id", "date_time"]
)

step_order = {
    "start": 0,
    "step_1": 1,
    "step_2": 2,
    "step_3": 3,
    "confirm": 4
}

error_data["current_order"] = error_data["process_step"].map(step_order)
error_data["next_step"] = (
    error_data.groupby("visit_id")["process_step"].shift(-1)
)
error_data["next_order"] = error_data["next_step"].map(step_order)

# A backward transition means the user moves to an earlier process step.
error_data["backward_transition"] = (
    error_data["next_order"] < error_data["current_order"]
)

error_rate = (
    error_data
    .dropna(subset=["next_order"])
    .groupby("Variation")["backward_transition"]
    .agg(
        total_steps="count",
        errors="sum"
    )
)

error_rate["error_rate_%"] = (
    error_rate["errors"]
    / error_rate["total_steps"]
    * 100
).round(2)

error_rate


In [ ]:
error_counts = error_rate["errors"].values
error_totals = error_rate["total_steps"].values

z_error, p_error = proportions_ztest(
    error_counts,
    error_totals
)

print("Error Rate Z-statistic:", round(z_error, 4))
print("Error Rate P-value:", p_error)


## 8. Client demographics and experiment balance

In [ ]:
experiment_demo = experiment_clients_ab.merge(
    demo,
    on="client_id",
    how="left"
)

print("Shape:", experiment_demo.shape)

print("\nAge by group:")
print(
    experiment_demo
    .groupby("Variation")["clnt_age"]
    .agg(["count", "mean", "median", "min", "max"])
)

print("\nTenure by group:")
print(
    experiment_demo
    .groupby("Variation")["clnt_tenure_yr"]
    .agg(["count", "mean", "median", "min", "max"])
)


In [ ]:
gender_counts = pd.crosstab(
    experiment_demo["Variation"],
    experiment_demo["gendr"]
)

gender_distribution = (
    pd.crosstab(
        experiment_demo["Variation"],
        experiment_demo["gendr"],
        normalize="index"
    ) * 100
).round(2)

print("Gender counts:")
print(gender_counts)

print("\nGender distribution (%):")
print(gender_distribution)


In [ ]:
# Chi-square test for gender balance.
chi2, p_gender, dof, expected = chi2_contingency(gender_counts)

print("Chi-square statistic:", round(chi2, 4))
print("P-value:", p_gender)


In [ ]:
# Welch's t-test for mean age.
control_age = experiment_demo.loc[
    experiment_demo["Variation"] == "Control",
    "clnt_age"
].dropna()

test_age = experiment_demo.loc[
    experiment_demo["Variation"] == "Test",
    "clnt_age"
].dropna()

t_age, p_age = ttest_ind(
    test_age,
    control_age,
    equal_var=False
)

print("Control mean age:", round(control_age.mean(), 2))
print("Test mean age:", round(test_age.mean(), 2))
print("T-statistic:", round(t_age, 4))
print("P-value:", p_age)


## 9. Tableau export files

In [ ]:
processed_path = "../data/processed"
os.makedirs(processed_path, exist_ok=True)

# 1. Conversion KPIs
tableau_kpis = ab_conversion.reset_index()
tableau_kpis.to_csv(
    os.path.join(processed_path, "tableau_kpis.csv"),
    index=False
)

# 2. Time by process step
tableau_time_by_step = average_time_by_step.copy()
tableau_time_by_step.to_csv(
    os.path.join(processed_path, "tableau_time_by_step.csv"),
    index=False
)

# 3. Error rate
tableau_error = error_rate.reset_index()
tableau_error.to_csv(
    os.path.join(processed_path, "tableau_error.csv"),
    index=False
)

# 4. Client-level demographics for Tableau filters
tableau_demographics = experiment_demo.copy()
tableau_demographics.to_csv(
    os.path.join(processed_path, "tableau_demographics.csv"),
    index=False
)

# 5. Funnel data
funnel = (
    web_with_variation
    .groupby(["Variation", "process_step"])["visit_id"]
    .nunique()
    .reset_index(name="unique_visits")
)

funnel["process_order"] = funnel["process_step"].map(step_order)
funnel = funnel.sort_values(["Variation", "process_order"])

tableau_funnel = funnel.drop(columns="process_order")
tableau_funnel.to_csv(
    os.path.join(processed_path, "tableau_funnel.csv"),
    index=False
)

print("Files created:")
for filename in sorted(os.listdir(processed_path)):
    if filename.startswith("tableau_") and filename.endswith(".csv"):
        print("-", filename)


## 10. Final findings

### Completion rate
The Test group achieved a higher completion rate than the Control group (**69.29% vs 65.59%**). The difference is statistically significant according to the two-proportion z-test.

### 5% improvement target
The Test group improved completion by approximately **5.65% relative to Control**, exceeding the numerical 5% threshold. However, the one-sided threshold test produced **p = 0.0644**, so the evidence is not statistically significant at the conventional 5% significance level.

### Time spent
The Test and Control versions show different time spent across process steps. The most notable difference is at the `confirm` step, where the Test group takes longer on average.

### Error rate
The Test group has a higher backward-navigation error rate (**9.25%**) than Control (**6.82%**). This difference is statistically significant.

### Experiment balance
The groups are not equal in size. Gender distribution does not show a statistically significant difference, while mean age differs statistically but only by about **0.34 years**, which is small in practical terms.

### Overall interpretation
The new design improves completion rate, but the higher error rate and longer time at some steps indicate usability trade-offs. The recommendation should therefore consider both conversion improvement and user experience rather than relying on completion rate alone.
